In [309]:
import numpy as np
import pandas as pd
import scipy
import math

In [310]:
water = pd.read_csv("properties_water.txt", sep=" ")
water_50bar = pd.read_csv("water_50bar.txt", sep="\t")
water_1d5bar = pd.read_csv("water_1.5bar.txt", sep="\t")

def find_val(df, colIn, colOut, valIn):
    #find value in property table, 
    # e.g. water_50bar, search T = 300k, what is heat capacity?
    idx = df[colIn][df[colIn]<=valIn].idxmax()
    inlow = df[colIn].iloc[idx]
    inhigh= df[colIn].iloc[idx+1]

    f = (valIn-inlow)/(inhigh-inlow) # how close the value is to its lower bound 
                                     #compared to higher bound
    Outlow = df[colOut].iloc[idx]
    Outhigh= df[colOut].iloc[idx+1]
    valOut = (1-f)*Outlow + f*Outhigh # linear (spline) interpolation
    return valOut

# Part 1: Heat exchanger only

In [311]:
Qdot = 50e3 #kJ/s
mdot = 150  #kg/s
CtoK = 273.15
Tin = CtoK+110 # K
Tcin = 110 # degC
deltaH = Qdot / mdot

Hin = find_val(water, "T|K", "hf|kJ/kg", Tin)
Tout = find_val(water, "hf|kJ/kg", "T|K", Hin-deltaH)
print("Influx (cold) temperature of district heating stream")
print(Tout - CtoK)

Influx (cold) temperature of district heating stream
30.562312599681036


In [312]:
#todo iterate on this, maybe, as it used the old assumption
Th, Tcguess = 240, 53.85 # This is set iteratively manually correct

H1 = find_val(water, "T|K", "hf|kJ/kg", CtoK+Th)
H2 = find_val(water, "T|K", "hf|kJ/kg", CtoK+Tcguess)
cpwell = (H1-H2)/(Th-Tcguess)
print("Cpavg, hot stream, 240 - Tcguess (~53) degC")
print(cpwell)

Cpavg, hot stream, 240 - Tcguess (~53) degC
4.362659145850121


In [313]:
print("Cpavg, cold stream, 110 - 30.5degC")
Cpcold=Qdot/mdot/(Tin-Tout)
Cpcold

Cpavg, cold stream, 110 - 30.5degC


4.19616109484067

In [314]:
#Eps-NTU method
#### Solve effectiveness, capacity flow rate for given NTU, q, Tcin,out, Thin, Th min. flow

Thin = 240
Tcout = Tout - CtoK
dTmax = Thin - Tcout


Cmax = mdot * Cpcold
print("Capacity flow max (district side)")
print(Cmax)

q = 50e3
NTU = 3
Cr = lambda eps : q/Cmax/eps/dTmax
root = lambda eps : 1/(q/Cmax/eps/dTmax-1)*np.log((eps-1)/(eps*q/Cmax/eps/dTmax-1))-NTU

eps_sol = scipy.optimize.fsolve(root,0.9)[0]

print("Efficiency")
print(eps_sol)

print("Capacity flow ratio, Cmin (geo brine side)")
Cratio = Cr(eps_sol)
print(Cratio)
print(Cratio*Cmax)
print("Mass flow rate geo brine side")
mdotGeo=mdot*Cratio*cpwell/Cpcold
print(mdotGeo)

print("Temperature out geo brine side")
Thout = Thin - eps_sol*(Thin-Tcout)
if abs(Tcguess-Thout) > 1:
    raise Exception("Initial guess for Thout / Tcguess not accurate enoyugh, please manually adjust")
else:
    print()
    print("Tcguess close enough to Thout")
    print()
print(Thout)
dT1 = Thin - Tcin
dT2 = Thout - Tcout
TLMTD = (dT2 - dT1)/math.log(dT2/dT1)
print("Logarithmic mean temperature difference")
print(TLMTD)

Capacity flow max (district side)
629.4241642261004
Efficiency
0.8888309862004026
Capacity flow ratio, Cmin (geo brine side)
0.426729406980949
268.5938003396833
Mass flow rate geo brine side
66.54921872962522
Temperature out geo brine side

Tcguess close enough to Thout

53.84529376044287
Logarithmic mean temperature difference
62.051568746519045


In [315]:
water_50bar.head()

,Temperature (K),Pressure (atm),Density (kg/m3),Volume (m3/kg),Internal Energy (kJ/kg),Enthalpy (kJ/kg),Entropy (J/g*K),Cv (J/g*K),Cp (J/g*K),Sound Spd. (m/s),Joule-Thomson (F/atm),Viscosity (Pa*s),Therm. Cond. (W/m*K),Phase
0,280.00,50.0,1002.3,0.000998,28.730,33.784,0.10386,4.1791,4.1811,1442.3,-0.042820,0.001428,0.57533,liquid
1,280.45,50.0,1002.3,0.000998,30.611,35.666,0.11058,4.1780,4.1804,1444.2,-0.042750,0.001409,0.57631,liquid
2,280.90,50.0,1002.3,0.000998,32.492,37.547,0.11728,4.1768,4.1797,1446.1,-0.042682,0.001390,0.57727,liquid
3,281.35,50.0,1002.2,0.000998,34.372,39.427,0.12397,4.1756,4.1791,1448.0,-0.042614,0.001371,0.57823,liquid
4,281.80,50.0,1002.2,0.000998,36.253,41.308,0.13065,4.1745,4.1784,1449.9,-0.042546,0.001353,0.57918,liquid


Density calculation

In [316]:
myseries = water_50bar.loc[water_50bar["Temperature (K)"].between(Thout+CtoK, Thin+CtoK), "Density (kg/m3)"]
print("Average density, between 240, ~53C, hot stream")
rhoGeo =sum(myseries)/len(myseries)
print(rhoGeo)

Average density, between 240, ~53C, hot stream
915.703188405797


In [317]:
myseries2 = water_1d5bar.loc[water_1d5bar["Temperature (K)"].between(Tcout+CtoK, Tcin+CtoK), "Density (kg/m3)"]
print("Average density, between 110, 30.5C, cold stream")
rhoDht = sum(myseries2)/len(myseries2)
print(rhoDht)

Average density, between 110, 30.5C, cold stream
976.119375


Viscosity calculation

In [318]:
myseries = water_50bar.loc[water_50bar["Temperature (K)"].between(Thout+CtoK, Thin+CtoK), "Viscosity (Pa*s)"]

print("Average viscosity, between 240, ~53C, hot stream")
etaGeo = sum(myseries)/len(myseries)
print(etaGeo)
#myseries

Average viscosity, between 240, ~53C, hot stream
0.0002232939371980676


In [319]:
myseries2 = water_1d5bar.loc[water_1d5bar["Temperature (K)"].between(Tcout+CtoK, Tcin+CtoK), "Viscosity (Pa*s)"]
print("Average viscosity, between 110, 30.5C, cold stream")
etaDht = sum(myseries2)/len(myseries2)
print(etaDht)

Average viscosity, between 110, 30.5C, cold stream
0.00043920225000000006


In [320]:
myseries = water_50bar.loc[water_50bar["Temperature (K)"].between(Thout+CtoK, Thin+CtoK), "Therm. Cond. (W/m*K)"]
print("heat conductivity, between 240, ~53C, hot stream")
lambdaGeo = sum(myseries)/len(myseries)
print(lambdaGeo)

heat conductivity, between 240, ~53C, hot stream
0.6695222463768116


In [321]:
myseries = water_1d5bar.loc[water_1d5bar["Temperature (K)"].between(Tcout+CtoK, Tcin+CtoK), "Therm. Cond. (W/m*K)"]
print("heat conductivity, between 110, 30.5C, cold stream")
lambdaDht = sum(myseries)/len(myseries)
print(lambdaDht)

heat conductivity, between 110, 30.5C, cold stream
0.655975


In [322]:
# Calculate Re = rho v d / eta
vel = 3 #m/s
d_in = 0.01
d_out = 0.012

flowGeo = mdotGeo / rhoGeo
flowDht = mdot    / rhoDht
AGeo = 0.25*math.pi*d_in**2
ADht = AGeo * flowDht / flowGeo
Peri = math.pi*d_out 
Lout = 4*ADht / Peri

ReGeo = rhoGeo * vel * d_in / etaGeo
ReDht = rhoDht * vel * Lout/ etaDht


print("Outer characteristic length 4A/P")
print(Lout)
print("Re_Geothermal, Re_Districtheat")
print(ReGeo,ReDht)

Outer characteristic length 4A/P
0.01762052503575096
Re_Geothermal, Re_Districtheat
123026.6078733984 117483.93286966819


In [323]:
# Calculate Nu, dittus boelter, Nu = 0.023Re^0.8Pr^n where n=0.3 cooling, n=0.4 heating
PrGeo = cpwell*1e3 * etaGeo / lambdaGeo
PrDht = Cpcold*1e3 * etaDht / lambdaDht

NuGeo = 0.023 * ReGeo**0.8 * PrGeo ** 0.3
NuDht = 0.023 * ReDht**0.8 * PrDht ** 0.4

print("Pr_Geothermal, Pr_Districtheat")
print(PrGeo,PrDht)
print("Nu_Geothermal, Nu_Districtheat")
print(NuGeo,NuDht)

Pr_Geothermal, Pr_Districtheat
1.4550007002781074 2.8095024874674888
Nu_Geothermal, Nu_Districtheat
303.7987876783845 395.51447788396786


In [324]:
# Calculate heat transfer coefficients based on Nu.
# h = Nu lambda / L
# Here, L is the inner diameter
# And the 4*Area / Perimeter

hGeo = NuGeo * lambdaGeo / d_in
hDht = NuDht * lambdaDht / Lout

print("heat transfer coefficients geo, dht")
print(hGeo,hDht)

heat transfer coefficients geo, dht
20340.004677298402 14724.17019944256


In [325]:
# Calculate steel heat transfer coef (inner area basis)
lambdaSteel = 15 #W/m/K

Usteel = lambdaSteel / (d_in*0.5) / math.log(d_out/d_in)
print("Heat transfer coefficient steel (inner diameter basis")
print(Usteel)

Heat transfer coefficient steel (inner diameter basis
16454.444843241236


In [326]:
# Calculate overall heat transfer coef
UTot = (1/hGeo+1/Usteel+d_in/(d_out*hDht))**-1
print("overall heat transfer coefficient")
print(UTot)

# Calculate heat ex. area
AreaTot = Qdot/(UTot/1e3)/TLMTD

print("total mass transfer area")
print(AreaTot)

#Calculate number of pipes
lenPipe = 6
APipe = lenPipe * d_in * math.pi
numPipes = AreaTot / APipe
print("Number of pipes")
print(numPipes)

overall heat transfer coefficient
6004.76774578303
total mass transfer area
134.19026932139488
Number of pipes
711.9014892444237


# Part 2: Calculate state variables

In [327]:
#Start with state 4. Saturated vapour.

# Assume a temperature approach / LMTD of 10K, so vapour at 230C = 503.15K
THigh=CtoK+230
pHigh = find_val(water, "T|K", "P|MPa", THigh)
enthal4 = find_val(water, "T|K", "hg|kJ/kg", THigh)
enthal3 = find_val(water, "T|K", "hf|kJ/kg", THigh)
entro4 = find_val(water, "T|K", "sg|kJ/kg/K", THigh)
entro3 = find_val(water, "T|K", "sf|kJ/kg/K", THigh)

#Isentropic, 80% steam quality. Make new column: 80% steam quality entropy.
water["s0.8|kJ/kg/K"]=0.2*water["sf|kJ/kg/K"]+0.8*water["sg|kJ/kg/K"]

p5 = find_val(water, "s0.8|kJ/kg/K", "P|MPa", entro4)
T5 = find_val(water, "s0.8|kJ/kg/K", "T|K", entro4)
enthal5f = find_val(water, "s0.8|kJ/kg/K", "hf|kJ/kg", entro4)
enthal5g = find_val(water, "s0.8|kJ/kg/K", "hg|kJ/kg", entro4)
enthal5=0.8*enthal5g+0.2*enthal5f

T1 = find_val(water, "P|MPa", "T|K", p5)
enthal1 = find_val(water, "T|K", "hf|kJ/kg", T1)
entro1 = find_val(water, "T|K", "sf|kJ/kg/K", T1)

#For state 2, use B2 slides: dH=vdp, incompressible fluid --> deltaH=deltap[kPa]/rho[kg/m3] yields [kJ/kg]
rhowT1 = find_val(water_1d5bar, "Temperature (K)", "Density (kg/m3)", T1)
#tables: p[MPa]
enthal2 = enthal1 + 1e3*(pHigh-p5)/rhowT1

print("The compressed pressure is (saturation pressure, 230C):")
print(pHigh)
print("The enthalpy at 4 is (sat. vapour, property table):")
print(enthal4)
print("The enthalpy at 3 is (sat. liquid, property table):")
print(enthal3)
print("The entropy at 4 is (sat. vapour, property table):")
print(entro4)
print("The entropy at 3 is (sat. liquid, property table):")
print(entro3)
print("The heat from 3->4 is equal to the enthalpy change:")
print(enthal4-enthal3)
print("Temperature, pressure, enthalpy at 5:")
print(T5-CtoK,p5,enthal5)
print("Work is then equal to enthalpy change 4->5")
print(enthal4-enthal5)
print("Pressure at 5->1 is equal. Temperature is found, sat liquid property table(should be equal to 5), as is enthalpy, entropy")
print(T1-CtoK, enthal1, entro1)
print("Heat is equal to enthalpy change 5->1")
print(enthal5-enthal1)
print("enthal2 = enthal1+(p2-p1)*(V/m)")
print(enthal2)
print("work is equal to enthalpy change 1->2")
print(enthal2-enthal1)
print("close the loop, heat equal to enthalpy change 2->3")
print(enthal3-enthal2)


The compressed pressure is (saturation pressure, 230C):
2.7992393699999987
The enthalpy at 4 is (sat. vapour, property table):
2802.8779999999997
The enthalpy at 3 is (sat. liquid, property table):
990.2097999999999
The entropy at 4 is (sat. vapour, property table):
6.212798
The entropy at 3 is (sat. liquid, property table):
2.6100429999999997
The heat from 3->4 is equal to the enthalpy change:
1812.6681999999998
Temperature, pressure, enthalpy at 5:
91.01997389033943 0.07272527023498711 2205.189072167102
Work is then equal to enthalpy change 4->5
597.6889278328977
Pressure at 5->1 is equal. Temperature is found, sat liquid property table(should be equal to 5), as is enthalpy, entropy
90.87964573510914 380.7386899618689 1.2029438905272656
Heat is equal to enthalpy change 5->1
1824.4503822052332
enthal2 = enthal1+(p2-p1)*(V/m)
383.56485509382935
work is equal to enthalpy change 1->2
2.8261651319604653
close the loop, heat equal to enthalpy change 2->3
606.6449449061705
